### **Universidad Autónoma de Occidente**
### *Facultad de Ingeniería y Ciencias Básicas*
### *Maestria en Inteligencia Artificial y Ciencia de Datos*
### **Curso**: Procesamiento de Datos secuenciales con Deep Learning
### **Profesor**: Natali Johana Velandia Fajardo
### **Estudiantes**:
* Soren Fabricius Acevedo (22500566).
* Ricardo Muñoz Bocanegra (22500246).
* Nicolas Lozano Mazuera (22500565).
* Juan José Bonilla Pinzón (22502052).
* Juan Manuel García Ortiz (22502268).



# Whisper — Procesamiento de Datos Secuenciales
**Paper:** Robust Speech Recognition via Large-Scale Weak Supervision (Radford et al., OpenAI 2022)

Este notebook tiene dos partes:
- **Parte 1:** demostracion del modelo en sus tres versiones: tiny, medium y turbo
- **Parte 2:** aplicacion Streamlit con visualizacion de arquitectura en tiempo real

## Instalacion de dependencias
Ejecutar una sola vez al inicio de la sesion en Colab.

In [ ]:
!pip install openai-whisper -q
!apt-get install -y ffmpeg -q
!pip install librosa soundfile -q
print('dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
dependencias instaladas


---
# PARTE 1 — Demostracion del modelo base

Cada seccion muestra el mismo audio procesado con un modelo distinto.
Se usa el flujo paso a paso para ver exactamente que hace Whisper internamente,
y luego la llamada simplificada transcribe() que hace todo en una linea.

### Audio de entrada
Cargar el archivo de audio antes de ejecutar las celdas de los modelos.

In [2]:
import whisper
import torch

# cargar el audio y recortarlo a 30 segundos
# whisper.load_audio: lee el archivo, convierte a float32 y remuestrea a 16kHz
# whisper.pad_or_trim: rellena con ceros o recorta exactamente a 480000 muestras (30s)
audio = whisper.load_audio('NUEVAYoL - Bad Bunny.mp3')
audio = whisper.pad_or_trim(audio)
print(f'audio cargado: {len(audio)} muestras a 16000 Hz = 30 segundos')

audio cargado: 480000 muestras a 16000 Hz = 30 segundos


---
## Modelo Tiny
- Parametros: 39M
- Capas encoder: 4 | Capas decoder: 4 | d_model: 384 | Cabezas: 6
- Uso: transcripcion rapida, bajo consumo de memoria, precision limitada

In [3]:
# cargar el modelo tiny con sus pesos preentrenados
# whisper.load_model descarga los pesos automaticamente si no estan en cache
model_tiny = whisper.load_model('tiny')

# whisper.log_mel_spectrogram: convierte el audio en espectrograma de Mel
# n_mels=model.dims.n_mels: usa el numero de bandas que espera este modelo (80)
# resultado: tensor de forma (80, 3000) -> 80 bandas x 3000 frames = 30 segundos
mel = whisper.log_mel_spectrogram(audio, n_mels=model_tiny.dims.n_mels).to(model_tiny.device)
print(f'tensor mel: {mel.shape} -> (bandas_mel, frames_temporales)')

# detect_language: pasa el mel por el encoder y predice el idioma
# devuelve la distribucion de probabilidad sobre 99 idiomas
_, probs = model_tiny.detect_language(mel)
idioma = max(probs, key=probs.get)
print(f'idioma detectado: {idioma} ({probs[idioma]*100:.1f}%)')

# whisper.DecodingOptions: parametros de decodificacion
# por defecto usa greedy decoding (elige el token mas probable en cada paso)
options = whisper.DecodingOptions()

# whisper.decode: ejecuta el decoder del Transformer
# Q del decoder consulta K y V del encoder en cada paso (Cross-Attention)
result = whisper.decode(model_tiny, mel, options)
print(f'transcripcion (tiny):')
print(result.text)

100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 180MiB/s]


tensor mel: torch.Size([80, 3000]) -> (bandas_mel, frames_temporales)
idioma detectado: es (86.1%)
transcripcion (tiny):
¡Muy mayor! Si te quieres ir al pie


In [4]:
# transcribe()
# internamente ejecuta los pasos: load_audio, log_mel_spectrogram,
# detect_language, y generacion autoregresiva token por token
result_tiny = model_tiny.transcribe('NUEVAYoL - Bad Bunny.mp3')
print('transcripcion completa (tiny):')
print(result_tiny['text'])

transcripcion completa (tiny):
 ¡ Tenn roasted la mayor! Y es que... ¡Mas de... ¡Une da lo enoe bailo! ¡Mue bailo! ¡Sue que... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Une... ¡Ura, dedo! ¡Ay, ay, ay, ay, ¿Vo a cada de ule, a pa' la y un ahí? ¡No con mi primavera, hecha de un loito ¡Míon el prón, sabe el la que hay ¡Con no te muy, por el buchito, muy ¡Vílico, no me dices vermolo ¡Ay, ¿Parte pasa los años y go dando cuáles? ¡Tendiendo dico, como cuadro, he un prido color ¡El perico es blanco! ¡Si, si, si, el tú si Rosita! ¡Ay, no te confundo! ¡No, no, no, más ahora vita! ¡Ay, un chate cañita en casa, un chate, un chate ¡Ay, para resescientes al quita! ¡Si, si, si! ¡Mueces también, no to nadie me lo vito! ¡Se ves a nuevo! ¡No me guande, ves a nuevo! ¡Ok, voy a estar rito! ¡Tomo guapón y vas a retercerpo! ¡Ay, ¿Pondres la tonida en po? ¡Ay, ¿Pondres la tonida en po? ¡Condres la tonida en po? ¡Pondres la tonida en po! ¡Pondres la tonida en po! ¡Mas en

---
## Modelo Medium
- Parametros: 769M
- Capas encoder: 24 | Capas decoder: 24 | d_model: 1024 | Cabezas: 16
- Uso: mayor precision, requiere GPU con al menos 5GB de VRAM

In [5]:
model_medium = whisper.load_model('medium')

mel = whisper.log_mel_spectrogram(audio, n_mels=model_medium.dims.n_mels).to(model_medium.device)
print(f'tensor mel: {mel.shape}')

_, probs = model_medium.detect_language(mel)
idioma = max(probs, key=probs.get)
print(f'idioma detectado: {idioma} ({probs[idioma]*100:.1f}%)')

options = whisper.DecodingOptions()
result = whisper.decode(model_medium, mel, options)
print(f'transcripcion (medium):')
print(result.text)

100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 87.2MiB/s]


tensor mel: torch.Size([80, 3000])
idioma detectado: es (78.7%)
transcripcion (medium):
Nueva York!


In [6]:
result_medium = model_medium.transcribe('NUEVAYoL - Bad Bunny.mp3')
print('transcripcion completa (medium):')
print(result_medium['text'])

transcripcion completa (medium):
 Nueva York! Si te quieres divertir con invento y con frinos Solo tienes que vivir Un verano en Nueva York Si te quieres divertir con invento y con frinos Solo tienes que vivir Un verano en Nueva York 4 de Julio, suena el u-lay Cuando te des de la cabeza 4 de Julio, suena el u-lay Ando con mi primo borracho de u-lay Los míos en el bronza, ven lo que hay Con lo no te mo' y por Washington mo' Willy Colón me dice en el mollo Porque pasan los años y sigo dando palo Tendiendo disco como cuadros fríos color El pelico es blanco, si si El Tusi es racista, eh eh No te confunda, no no Mejor evita, ey Un chat de cañita en casa de toñita Y PR se siente cerquita Si si si Como el campeonato nadie me lo quito The best in the world No me wonder, best in the world, ok? Puerto Rico Como va Bonnie, va a ser rave de hip-hop, ey Con reggaeton y dembow, ey Con reggaeton y dembow, si Con reggaeton y dembow Como va Bonnie, va a ser rave de hip-hop, ey Con reggaeton y dembow Me

---
## Modelo Turbo
- Parametros: 809M (large-v3 destilado)
- Capas encoder: 32 | Capas decoder: 4 | d_model: 1280 | Cabezas: 20
- Uso: velocidad similar a small, calidad cercana a large-v3
- Nota: turbo reduce las capas del decoder de 32 a 4 para ganar velocidad

In [7]:
model_turbo = whisper.load_model('turbo')

mel = whisper.log_mel_spectrogram(audio, n_mels=model_turbo.dims.n_mels).to(model_turbo.device)
print(f'tensor mel: {mel.shape}')

_, probs = model_turbo.detect_language(mel)
idioma = max(probs, key=probs.get)
print(f'idioma detectado: {idioma} ({probs[idioma]*100:.1f}%)')

options = whisper.DecodingOptions()
result = whisper.decode(model_turbo, mel, options)
print(f'transcripcion (turbo):')
print(result.text)

100%|█████████████████████████████████████| 1.51G/1.51G [00:23<00:00, 68.8MiB/s]


tensor mel: torch.Size([128, 3000])
idioma detectado: es (99.9%)
transcripcion (turbo):
¡Nueva York! Si te quieres divertir Con inverno y con primor


In [8]:
result_turbo = model_turbo.transcribe('NUEVAYoL - Bad Bunny.mp3')
print('transcripcion completa (turbo):')
print(result_turbo['text'])

transcripcion completa (turbo):
 ¡Nueva York! Si te quieres divertir Con inverno y con primor Solo tienes que vivir ¿A dónde? Un verano en Nueva York ¡Nueva York! Si te quieres divertir Con inverno y con primor ¿Pero qué es esto? Solo tienes que vivir ¿Y este frío? Un verano en Nueva York Un ratito, mamá Ey, ey, ey Otro de julio, fue el de July Ando con mi primo borracho, Duloy Los míos en el bronz Saben lo que hay Con la no tengo hoy Por Washington, hoy Willy Colón me dicen el molo Ey, porque pasan los años Y sigo dando palos Vendiendo discos Como cuadro Frida Kahlo El perico es blanco Si, si El Tusi racista Eh, eh No te confundas No, no Mejor evita Ey Un chat de cañita En casa de Toñita Y perro se siente cerquita Si, si, si Con el campeonato nadie me lo quita Debece no vos No me wande Debece no vos Ok Puerto Rico Como va Bonnie Va a ser rey del pop Ey Con reggaeton y dembow Ey Con reggaeton y dembow Si, con reggaeton y dembow Si, con reggaeton y dembow Como va Bonnie Va a ser rey del

---
# PARTE 2 — Aplicacion Streamlit

La app muestra la arquitectura completa de Whisper de forma visual
mientras procesa el audio en tiempo real.

Ejecutar las celdas en orden: instalacion -> escribir app.py -> token ngrok -> lanzar.

In [9]:
!pip install streamlit pyngrok -q
print('streamlit y pyngrok instalados')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 131.9 MB/s eta 0:00:00
streamlit y pyngrok instalados


In [10]:
%%writefile app.py
# app.py — Interfaz Streamlit para Whisper
# Materia: Procesamiento de Datos Secuenciales
# Muestra la arquitectura completa mientras procesa el audio

import streamlit as st
import whisper
import torch
import numpy as np
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import torch.nn.functional as F
import tempfile
import os
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(
    page_title='Whisper — Arquitectura y Transcripcion',
    layout='wide',
    initial_sidebar_state='expanded',
)

# ── tabla de parametros por modelo ──────────────────────
MODEL_SPECS = {
    'tiny':   {'params': '39M',  'enc_layers': 4,  'dec_layers': 4,  'd_model': 384,  'heads': 6,  'desc': 'Rapido, bajo consumo, menor precision'},
    'base':   {'params': '74M',  'enc_layers': 6,  'dec_layers': 6,  'd_model': 512,  'heads': 8,  'desc': 'Balance velocidad / calidad'},
    'small':  {'params': '244M', 'enc_layers': 12, 'dec_layers': 12, 'd_model': 768,  'heads': 12, 'desc': 'Buena precision, requiere mas memoria'},
    'medium': {'params': '769M', 'enc_layers': 24, 'dec_layers': 24, 'd_model': 1024, 'heads': 16, 'desc': 'Alta precision, requiere GPU 5GB+'},
    'turbo':  {'params': '809M', 'enc_layers': 32, 'dec_layers': 4,  'd_model': 1280, 'heads': 20, 'desc': 'large-v3 destilado: calidad alta, decoder reducido'},
}

# ── funcion para dibujar la arquitectura ─────────────────
def draw_architecture(paso_activo=None, resultado=None):
    '''
    Dibuja el diagrama de arquitectura de Whisper.
    paso_activo: nombre del paso que se esta ejecutando actualmente
    pasos completados se muestran en verde, pendientes en gris
    '''
    PASOS = [
        'audio',
        'mel',
        'conv',
        'pos_enc',
        'encoder',
        'hidden',
        'tokens',
        'decoder',
        'softmax',
        'salida',
    ]
    LABELS = {
        'audio':   'AUDIO\nEntrada',
        'mel':     'Log-Mel\nSpectrogram\n(80 x 3000)',
        'conv':    '2x Conv1D\n+ GELU',
        'pos_enc': 'Positional\nEncoding\nSinusoidal',
        'encoder': 'ENCODER\nN x [Self-Attn\n+ LayerNorm\n+ FFN]',
        'hidden':  'Hidden\nStates\n(1500 x d)',
        'tokens':  'Tokens de\nControl\n[SOT][idioma]\n[tarea]',
        'decoder': 'DECODER\nN x [Masked Self-Attn\n+ Cross-Attn\n+ FFN]',
        'softmax': 'Linear\n+ Softmax',
        'salida':  'TEXTO\nFinal',
    }
    NOTAS = {
        'audio':   '',
        'mel':     'n_fft=400 hop=160',
        'conv':    'extrae features locales',
        'pos_enc': 'codifica posicion temporal',
        'encoder': 'Q, K, V del mismo audio',
        'hidden':  'representacion densa del audio',
        'tokens':  '<|startoftranscript|>',
        'decoder': 'Q<-dec  K,V<-encoder (Cross-Attn)',
        'softmax': 'vocabulario: 51865 tokens',
        'salida':  '',
    }

    idx_activo = PASOS.index(paso_activo) if paso_activo in PASOS else -1

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#0f1117')

    # posiciones x de las dos columnas
    xs = [2.5, 7.5]
    # posiciones y de los pasos (5 arriba, 5 abajo)
    ys = [8.8, 7.2, 5.8, 4.4, 3.0]

    # coordenadas de cada paso
    coords = {
        'audio':   (xs[0], ys[0]),
        'mel':     (xs[0], ys[1]),
        'conv':    (xs[0], ys[2]),
        'pos_enc': (xs[0], ys[3]),
        'encoder': (xs[0], ys[4]),
        'hidden':  (xs[1], ys[4]),
        'tokens':  (xs[1], ys[3]),
        'decoder': (xs[1], ys[2]),
        'softmax': (xs[1], ys[1]),
        'salida':  (xs[1], ys[0]),
    }

    for i, paso in enumerate(PASOS):
        cx, cy = coords[paso]
        if i < idx_activo:
            bg, fg, ec, lw = '#1a472a', '#a6e3a1', '#a6e3a1', 2.0  # completado
        elif i == idx_activo:
            bg, fg, ec, lw = '#1e3a5f', '#89b4fa', '#89b4fa', 2.5  # activo
        else:
            bg, fg, ec, lw = '#1c1c2e', '#6c7086', '#45475a', 1.0  # pendiente

        # caja del paso
        box_w, box_h = 2.2, 1.1
        rect = mpatches.FancyBboxPatch(
            (cx - box_w/2, cy - box_h/2), box_w, box_h,
            boxstyle='round,pad=0.05', linewidth=lw,
            facecolor=bg, edgecolor=ec,
        )
        ax.add_patch(rect)
        ax.text(cx, cy, LABELS[paso], ha='center', va='center',
                fontsize=7.5, color=fg, fontweight='bold' if i == idx_activo else 'normal',
                multialignment='center')

        # nota al costado
        if NOTAS[paso]:
            nota_x = cx + 1.3 if cx < 5 else cx - 1.3
            ha = 'left' if cx < 5 else 'right'
            ax.text(nota_x, cy, NOTAS[paso], ha=ha, va='center',
                    fontsize=6.5, color='#585b70', style='italic')

    # flechas verticales columna izquierda
    left_order = ['audio', 'mel', 'conv', 'pos_enc', 'encoder']
    for a, b in zip(left_order, left_order[1:]):
        x0, y0 = coords[a]
        x1, y1 = coords[b]
        ax.annotate('', xy=(x1, y1 + 0.58), xytext=(x0, y0 - 0.58),
                    arrowprops=dict(arrowstyle='->', color='#585b70', lw=1.2))

    # flecha horizontal encoder -> hidden (cross-attention bridge)
    ax.annotate('', xy=(coords['hidden'][0] - 1.1, coords['hidden'][1]),
                xytext=(coords['encoder'][0] + 1.1, coords['encoder'][1]),
                arrowprops=dict(arrowstyle='->', color='#f38ba8', lw=2.0))
    ax.text(5.0, ys[4] + 0.35, 'Cross-Attention', ha='center', va='center',
            fontsize=7, color='#f38ba8', fontweight='bold')
    ax.text(5.0, ys[4] + 0.08, 'K, V <- encoder  Q <- decoder', ha='center', va='center',
            fontsize=6.5, color='#f38ba8')

    # flechas verticales columna derecha (de abajo hacia arriba)
    right_order = ['hidden', 'tokens', 'decoder', 'softmax', 'salida']
    for a, b in zip(right_order, right_order[1:]):
        x0, y0 = coords[a]
        x1, y1 = coords[b]
        ax.annotate('', xy=(x1, y1 - 0.58), xytext=(x0, y0 + 0.58),
                    arrowprops=dict(arrowstyle='->', color='#585b70', lw=1.2))

    # etiquetas de columna
    ax.text(xs[0], 9.7, 'ENCODER', ha='center', fontsize=11, color='#74c7ec',
            fontweight='bold')
    ax.text(xs[1], 9.7, 'DECODER', ha='center', fontsize=11, color='#cba6f7',
            fontweight='bold')

    # si hay resultado, mostrar el texto final en el paso salida
    if resultado and paso_activo == 'salida':
        texto_corto = resultado[:60] + '...' if len(resultado) > 60 else resultado
        ax.text(xs[1], ys[0] - 0.85, '"' + texto_corto + '"',
                ha='center', va='top', fontsize=7, color='#a6e3a1',
                wrap=True, style='italic')

    plt.tight_layout(pad=0.2)
    return fig


# ── sidebar ──────────────────────────────────────────────
with st.sidebar:
    st.title('Configuracion')
    st.markdown('---')
    model_size = st.selectbox(
        'Modelo de Whisper:',
        options=list(MODEL_SPECS.keys()),
        index=1,
    )
    spec = MODEL_SPECS[model_size]
    st.caption(spec['desc'])
    st.markdown('---')
    st.markdown('**Parametros del modelo**')
    st.markdown(f'- Parametros totales: {spec["params"]}')
    st.markdown(f'- Capas encoder: {spec["enc_layers"]}')
    st.markdown(f'- Capas decoder: {spec["dec_layers"]}')
    st.markdown(f'- d_model: {spec["d_model"]}')
    st.markdown(f'- Cabezas de atencion: {spec["heads"]}')
    st.markdown('---')
    language_option = st.selectbox(
        'Idioma del audio:',
        ['Autodetectar', 'Espanol (es)', 'Ingles (en)', 'Frances (fr)', 'Portugues (pt)'],
    )
    lang_map = {
        'Autodetectar': None, 'Espanol (es)': 'es',
        'Ingles (en)': 'en', 'Frances (fr)': 'fr', 'Portugues (pt)': 'pt',
    }
    selected_lang = lang_map[language_option]
    st.markdown('---')
    st.markdown('[Paper arXiv:2212.04356](https://arxiv.org/abs/2212.04356)')
    st.markdown('[Repositorio openai/whisper](https://github.com/openai/whisper)')

# ── titulo ───────────────────────────────────────────────
st.title('Whisper — Arquitectura Transformer Encoder-Decoder')
st.markdown(
    'Paper: *Robust Speech Recognition via Large-Scale Weak Supervision* (OpenAI, 2022)  \n'
    'Modelo seleccionado: **' + model_size + '** — ' + MODEL_SPECS[model_size]['desc']
)
st.markdown('---')

# ── tabs ─────────────────────────────────────────────────
tab_demo, tab_arq, tab_specs = st.tabs([
    'Transcripcion en Vivo',
    'Arquitectura de Whisper',
    'Comparativa de Modelos',
])

# ══════════════════════════════════════════════════════════
# TAB 1 — TRANSCRIPCION CON ARQUITECTURA EN TIEMPO REAL
# ══════════════════════════════════════════════════════════
with tab_demo:
    st.header('Transcripcion con Visualizacion de Arquitectura')
    st.markdown(
        'A medida que Whisper procesa el audio, la arquitectura se ilumina paso a paso. '
        'El paso activo aparece en azul, los completados en verde.'
    )

    col_up, col_btn = st.columns([3, 1])
    with col_up:
        uploaded_file = st.file_uploader(
            'Cargar archivo de audio',
            type=['wav', 'mp3', 'm4a', 'ogg', 'flac', 'webm'],
        )
    with col_btn:
        st.markdown('<br>', unsafe_allow_html=True)
        transcribe_btn = st.button('Transcribir', type='primary', disabled=(uploaded_file is None))

    # layout: arquitectura a la izquierda, resultados a la derecha
    col_arq, col_res = st.columns([1, 1])

    with col_arq:
        st.subheader('Flujo de procesamiento')
        arq_placeholder = st.empty()
        # mostrar arquitectura en estado inicial (todo pendiente)
        arq_placeholder.pyplot(draw_architecture(None))
        plt.close('all')

    with col_res:
        st.subheader('Resultados')
        status_box   = st.empty()
        mel_box      = st.empty()
        result_box   = st.empty()
        segments_box = st.empty()
        attn_box     = st.empty()

    if uploaded_file is not None and transcribe_btn:
        with tempfile.NamedTemporaryFile(delete=False,
                suffix=os.path.splitext(uploaded_file.name)[1]) as tmp:
            tmp.write(uploaded_file.read())
            tmp_path = tmp.name

        try:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

            # ── PASO 1: audio entrada ──────────────────────
            arq_placeholder.pyplot(draw_architecture('audio'))
            plt.close('all')
            status_box.info('Cargando audio...')

            audio_signal, sr_orig = librosa.load(tmp_path, sr=None, mono=True)
            dur = len(audio_signal) / sr_orig
            col1, col2, col3 = st.columns(3)
            col1.metric('Duracion', f'{dur:.1f} s')
            col2.metric('Sample rate original', f'{sr_orig:,} Hz')
            col3.metric('Dispositivo', device.upper())

            # ── PASO 2: log-mel spectrogram ────────────────
            arq_placeholder.pyplot(draw_architecture('mel'))
            plt.close('all')
            status_box.info('Generando espectrograma de Mel...')

            SAMPLE_RATE = 16000
            HOP_LENGTH  = 160
            N_MELS      = 80
            audio_16k = librosa.resample(audio_signal, orig_sr=sr_orig, target_sr=SAMPLE_RATE) if sr_orig != SAMPLE_RATE else audio_signal
            mel_spec  = librosa.feature.melspectrogram(
                y=audio_16k, sr=SAMPLE_RATE, n_fft=400,
                hop_length=HOP_LENGTH, n_mels=N_MELS, fmax=8000
            )
            mel_db = librosa.power_to_db(mel_spec, ref=np.max)

            fig_mel, ax_mel = plt.subplots(figsize=(7, 2.5))
            fig_mel.patch.set_facecolor('#0f1117')
            ax_mel.set_facecolor('#0f1117')
            librosa.display.specshow(
                mel_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                x_axis='time', y_axis='mel', fmax=8000,
                ax=ax_mel, cmap='magma'
            )
            ax_mel.set_title(f'Log-Mel Spectrogram — tensor ({mel_db.shape[0]}, {mel_db.shape[1]})',
                             fontsize=9, color='#cdd6f4')
            ax_mel.tick_params(colors='#6c7086')
            plt.tight_layout()
            mel_box.pyplot(fig_mel)
            plt.close('all')

            # ── PASO 3: conv1d ─────────────────────────────
            arq_placeholder.pyplot(draw_architecture('conv'))
            plt.close('all')
            status_box.info('Aplicando Conv1D...')

            # ── PASO 4: positional encoding ────────────────
            arq_placeholder.pyplot(draw_architecture('pos_enc'))
            plt.close('all')
            status_box.info('Positional encoding sinusoidal...')

            # ── PASO 5: encoder ────────────────────────────
            arq_placeholder.pyplot(draw_architecture('encoder'))
            plt.close('all')
            status_box.info(f'Cargando modelo {model_size}...')

            @st.cache_resource
            def load_whisper_model(size, dev):
                return whisper.load_model(size, device=dev)

            model = load_whisper_model(model_size, device)

            # ── PASO 6: hidden states ──────────────────────
            arq_placeholder.pyplot(draw_architecture('hidden'))
            plt.close('all')
            status_box.info('Encoder generando hidden states...')

            # deteccion de idioma (pasa el mel por el encoder)
            audio_w  = whisper.load_audio(tmp_path)
            audio_pad = whisper.pad_or_trim(audio_w)
            mel_t    = whisper.log_mel_spectrogram(audio_pad).to(device)
            _, probs = model.detect_language(mel_t)
            idioma   = selected_lang if selected_lang else max(probs, key=probs.get)

            # ── PASO 7: tokens de control ──────────────────
            arq_placeholder.pyplot(draw_architecture('tokens'))
            plt.close('all')
            status_box.info(f'Idioma: {idioma} — preparando tokens de control...')

            # ── PASO 8: decoder ────────────────────────────
            arq_placeholder.pyplot(draw_architecture('decoder'))
            plt.close('all')
            status_box.info('Decoder generando tokens...')

            # captura de cross-attention con hooks
            cross_attn_data = {}
            def make_hook(name):
                def fn(module, inp, out):
                    if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
                        cross_attn_data[name] = out[1].detach().cpu()
                return fn
            hooks = [
                blk.cross_attn.register_forward_hook(make_hook(f'blk{i}'))
                for i, blk in enumerate(model.decoder.blocks)
            ]
            with torch.no_grad():
                resultado = model.transcribe(
                    tmp_path,
                    language=idioma,
                    verbose=False,
                    fp16=(device == 'cuda'),
                )
            for h in hooks:
                h.remove()

            # ── PASO 9: softmax ────────────────────────────
            arq_placeholder.pyplot(draw_architecture('softmax'))
            plt.close('all')
            status_box.info('Linear + Softmax...')

            # ── PASO 10: salida final ──────────────────────
            arq_placeholder.pyplot(draw_architecture('salida', resultado['text']))
            plt.close('all')
            status_box.success(f'Completado — idioma: {resultado["language"].upper()}')

            # resultado
            result_box.success(resultado['text'].strip())

            # tabla de segmentos
            if resultado['segments']:
                rows = [
                    {
                        'Seg': i + 1,
                        'Inicio': f'{s["start"]:.1f}s',
                        'Fin':    f'{s["end"]:.1f}s',
                        'Texto':  s['text'].strip(),
                    }
                    for i, s in enumerate(resultado['segments'])
                ]
                segments_box.dataframe(rows, use_container_width=True)

            # heatmap cross-attention
            if cross_attn_data:
                key0  = list(cross_attn_data.keys())[0]
                attn  = cross_attn_data[key0][0].numpy()  # (n_heads, n_text, n_audio)
                n_show = min(4, attn.shape[0])
                fig_a, axes_a = plt.subplots(1, n_show, figsize=(4 * n_show, 3))
                fig_a.patch.set_facecolor('#0f1117')
                if n_show == 1:
                    axes_a = [axes_a]
                fig_a.suptitle('Cross-Attention — Q del decoder, K y V del encoder',
                               fontsize=9, color='#cdd6f4')
                for hi in range(n_show):
                    axes_a[hi].imshow(attn[hi], cmap='hot', aspect='auto')
                    axes_a[hi].set_title(f'Cabeza {hi+1}', fontsize=8, color='#cdd6f4')
                    axes_a[hi].set_xlabel('Frames audio (encoder)', fontsize=7, color='#6c7086')
                    axes_a[hi].set_ylabel('Tokens texto (decoder)', fontsize=7, color='#6c7086')
                    axes_a[hi].tick_params(colors='#45475a')
                plt.tight_layout()
                attn_box.pyplot(fig_a)
                plt.close('all')

        finally:
            os.unlink(tmp_path)

    elif uploaded_file is None:
        status_box.info('Sube un archivo de audio para comenzar.')

# ══════════════════════════════════════════════════════════
# TAB 2 — ARQUITECTURA ESTATICA EXPLICADA
# ══════════════════════════════════════════════════════════
with tab_arq:
    st.header('Arquitectura de Whisper — Sequence to Sequence')

    st.markdown('''
El flujo completo de Whisper sigue el patron de un Transformer encoder-decoder.
Cada componente tiene una funcion especifica en la cadena de procesamiento.
''')

    st.subheader('Encoder — procesamiento del audio')
    pasos_enc = [
        ('Log-Mel Spectrogram', 'El audio se remuestrea a 16kHz. Luego se aplica STFT con ventanas de 25ms (n_fft=400) y salto de 10ms (hop=160). Se aplica un banco de 80 filtros Mel que imitan la percepcion logaritmica del oido humano. Resultado: tensor (80, 3000) para 30 segundos.'),
        ('2x Conv1D + GELU', 'Dos capas convolucionales con activacion GELU extraen patrones locales del espectrograma antes de que entre al Transformer. Reducen los 3000 frames a 1500 y proyectan a d_model dimensiones.'),
        ('Positional Encoding Sinusoidal', 'El Transformer procesa todas las posiciones en paralelo, por lo que no sabe cual frame viene primero. Se suma una senal sinusoidal unica a cada posicion para codificar el orden temporal. En Whisper esta senal es fija (no se aprende).'),
        ('N x [Self-Attention + LayerNorm + FFN]', 'Cada frame del audio puede atender a todos los demas frames (Self-Attention). Los pesos Q, K, V se calculan como proyecciones lineales de la entrada. Con 8 cabezas en base, el modelo aprende 8 tipos distintos de relaciones en paralelo. La salida son los hidden states.'),
    ]
    for titulo, desc in pasos_enc:
        with st.expander(titulo):
            st.markdown(desc)

    st.subheader('Puente — Cross-Attention')
    st.markdown('''
**Q viene del decoder, K y V vienen del encoder.**

El decoder genera una consulta (Q) que representa 'que necesito saber del audio para generar el siguiente token'.
El encoder responde con sus K y V que representan el audio ya procesado.
La formula es: Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V
Sin Cross-Attention, el decoder generaria texto sin ninguna relacion con el audio.
''')

    st.subheader('Decoder — generacion de texto')
    pasos_dec = [
        ('Tokens de Control (Multitasking)', 'El decoder empieza con tokens especiales que definen la tarea: [startoftranscript] marca el inicio, [idioma] le dice el idioma del audio, [transcribe] o [translate] le dice si debe transcribir o traducir. Cambiando solo estos tokens se obtienen comportamientos distintos sin modificar los pesos del modelo.'),
        ('Learned Positional Encoding', 'A diferencia del encoder, el decoder usa embeddings posicionales aprendidos durante el entrenamiento, no fijos.'),
        ('Masked Self-Attention', 'El decoder ve los tokens de texto ya generados, pero con una mascara causal que impide ver tokens futuros. Esto garantiza que la generacion en entrenamiento sea identica a la de inferencia.'),
        ('Cross-Attention (Q <- decoder, K/V <- encoder)', 'En cada bloque del decoder, el estado actual del decoder consulta los hidden states del encoder. Es el mecanismo que conecta el audio con la generacion de texto.'),
        ('FFN + Linear + Softmax', 'Cada bloque termina con una red Feed-Forward. Al final, una capa lineal proyecta al tamano del vocabulario (51865 tokens) y softmax convierte en probabilidades. Se elige el token mas probable.'),
    ]
    for titulo, desc in pasos_dec:
        with st.expander(titulo):
            st.markdown(desc)

# ══════════════════════════════════════════════════════════
# TAB 3 — COMPARATIVA DE MODELOS
# ══════════════════════════════════════════════════════════
with tab_specs:
    st.header('Comparativa de Modelos de Whisper')

    tabla = []
    for nombre, s in MODEL_SPECS.items():
        tabla.append({
            'Modelo':           nombre,
            'Parametros':       s['params'],
            'Capas encoder':    s['enc_layers'],
            'Capas decoder':    s['dec_layers'],
            'd_model':          s['d_model'],
            'Cabezas atencion': s['heads'],
            'Descripcion':      s['desc'],
        })
    st.dataframe(tabla, use_container_width=True)

    st.markdown('---')
    st.subheader('Nota sobre el modelo Turbo')
    st.markdown('''
Turbo es una destilacion de large-v3. Mantiene el encoder de 32 capas (d_model=1280)
pero reduce el decoder de 32 a solo 4 capas. Esto lo hace significativamente mas rapido
sin perder mucha calidad en transcripcion. Es el modelo recomendado cuando se necesita
alta precision pero con restricciones de tiempo de inferencia.
''')

    st.subheader('Innovaciones del paper (Section 2)')
    inn = [
        ('Escala de entrenamiento', '680,000 horas de audio de internet con supervision debil, versus 1,000 horas de los modelos anteriores.'),
        ('Multitarea con tokens especiales', 'Un solo modelo hace transcripcion, traduccion y deteccion de silencio cambiando solo los tokens de control al inicio del decoder.'),
        ('Zero-shot en 99 idiomas', 'Sin fine-tuning por idioma. En ingles alcanza WER de 2.7%, comparable con sistemas especializados.'),
    ]
    for titulo, desc in inn:
        with st.expander(titulo):
            st.markdown(desc)

    st.subheader('Limitaciones documentadas')
    lim = [
        ('Alucinaciones en silencio', 'Seccion 4.5 del paper — con ruido puro o silencio el decoder puede generar texto que no existe en el audio.'),
        ('Rendimiento desigual por idioma', 'Figura 3 del paper — idiomas con menos datos en internet tienen mayor tasa de error.'),
        ('Sin memoria entre bloques de 30s', 'Seccion 2.4 del paper — cada bloque es independiente, puede perder contexto en audios largos.'),
        ('Costo computacional', 'Tabla 1 del paper — large necesita ~10GB de VRAM.'),
    ]
    for titulo, ref in lim:
        with st.expander(titulo):
            st.markdown(ref)

Writing app.py


### Configurar token de ngrok
Obtener el token en https://dashboard.ngrok.com/get-started/your-authtoken y reemplazar TU_TOKEN_AQUI.

In [11]:
from pyngrok import ngrok

NGROK_TOKEN = '3BJ7erpGC4XQsaOMLIf8cE3pKOm_4Ntsmw6u2rqoHraroGwXK'
ngrok.set_auth_token(NGROK_TOKEN)
print('token configurado')

token configurado


### Lanzar Streamlit

In [12]:
import subprocess, time
from pyngrok import ngrok
from IPython.display import display, HTML

proceso = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)
tunnel = ngrok.connect(8501, 'http')
url    = tunnel.public_url

print('URL publica:')
print(url)
display(HTML(f'<a href="{url}" target="_blank" style="font-size:16px">Abrir app</a>'))

URL publica:
https://unintroversive-raye-hysteretic.ngrok-free.dev


### Detener la app (opcional)

In [ ]:
proceso.terminate()
ngrok.kill()
print('app detenida')